In [1]:
print("hello")

hello


In [ ]:
# uv pip install psycopg2
# uv pip install neo4j

Note: you may need to restart the kernel to use updated packages.


Using Python 3.11.9 environment at: c:\Users\remote\Documents\pyspark-nov-proj\.venv
Resolved 2 packages in 81ms
Installed 1 package in 35ms
 + neo4j==6.2.0


In [7]:
import psycopg2
from neo4j import GraphDatabase

# 1. Connect to 

In [19]:
pg_conn = psycopg2.connect(
    dbname="bank_db", user="postgres", password="root", host="localhost", port="5432"
)
pg_cursor = pg_conn.cursor()

pg_cursor.execute("""
SELECT 
    c.table_name, 
    c.column_name, 
    c.data_type,
    -- Identify if the column is a Primary Key (PK) or Foreign Key (FK)
    CASE 
        WHEN tc.constraint_type = 'PRIMARY KEY' THEN 'PK'
        WHEN tc.constraint_type = 'FOREIGN KEY' THEN 'FK'
        ELSE 'NONE'
    END AS key_type,
    -- If it is a Foreign Key, pull the table and column it points to
    ccu.table_name AS referenced_table,
    ccu.column_name AS referenced_column
FROM 
    information_schema.columns c
LEFT JOIN 
    information_schema.key_column_usage kcu 
    ON c.table_schema = kcu.table_schema 
    AND c.table_name = kcu.table_name 
    AND c.column_name = kcu.column_name
LEFT JOIN 
    information_schema.table_constraints tc 
    ON kcu.table_schema = tc.table_schema 
    AND kcu.table_name = tc.table_name 
    AND kcu.constraint_name = tc.constraint_name
LEFT JOIN 
    information_schema.constraint_column_usage ccu 
    ON tc.constraint_type = 'FOREIGN KEY' 
    AND tc.constraint_name = ccu.constraint_name
WHERE 
    c.table_schema = 'public'
ORDER BY 
    c.table_name, 
    c.ordinal_position;

""")
metadata = pg_cursor.fetchall() # Returns list of tuples: (table, column, type)

# 2. Connect to Neo4j and ingest data
neo4j_driver = GraphDatabase.driver("bolt://localhost:7687", auth=("neo4j", "password123"))

cypher_query = """
UNWIND $rows AS row

// 1. Create or match the parent Table node
MERGE (t:Table {name: row.table_name})

// 2. Create a unique Column node scoped to its parent Table to avoid naming collisions
// (e.g., 'id' columns in different tables remain distinct)
MERGE (c:Column {id: row.table_name + '.' + row.column_name})
SET c.name = row.column_name, 
    c.dataType = row.data_type

// 3. Connect the Table to its Column
MERGE (t)-[:HAS_COLUMN]->(c)

// 4. If it is a Primary Key, tag the Column node dynamically
FOREACH (_ IN CASE WHEN row.key_type = 'PK' THEN [1] ELSE [] END |
    SET c.isPrimaryKey = true
)

// 5. If it is a Foreign Key, draw a direct data-line relationship between the Tables
FOREACH (_ IN CASE WHEN row.key_type = 'FK' AND row.referenced_table IS NOT NULL THEN [1] ELSE [] END |
    MERGE (targetTable:Table {name: row.referenced_table})
    MERGE (t)-[r:REFERENCES]->(targetTable)
    SET r.from_column = row.column_name, 
        r.to_column = row.referenced_column
)
"""


# Format data payload for Neo4j efficiency (mapping all 6 SQL return values)
batch_data = [
    {
        "table_name": r[0],
        "column_name": r[1],
        "data_type": r[2],
        "key_type": r[3],
        "referenced_table": r[4],
        "referenced_column": r[5]
    } 
    for r in metadata
]

with neo4j_driver.session() as session:
    session.run(cypher_query, rows=batch_data)

# Clean up connections
pg_cursor.close()
pg_conn.close()
neo4j_driver.close()
print("Metadata successfully loaded to Neo4j!")


Metadata successfully loaded to Neo4j!


In [12]:
cypher_query

'\nUNWIND $rows AS row\nMERGE (t:Table {name: row.table_name})\nMERGE (c:Column {name: row.column_name, dataType: row.data_type})\nMERGE (t)-[:HAS_COLUMN]->(c)\n'

In [15]:
metadata

[('accounts', 'accountnumber', 'integer', 'PK', None, None),
 ('accounts', 'customerid', 'integer', 'FK', 'customer', 'customerid'),
 ('accounts', 'accountopendate', 'date', 'NONE', None, None),
 ('accounts', 'branchcode', 'character varying', 'FK', 'branch', 'branchcode'),
 ('accounts', 'accountstatus', 'character varying', 'NONE', None, None),
 ('accounts', 'balance', 'numeric', 'NONE', None, None),
 ('branch', 'branchcode', 'character varying', 'PK', None, None),
 ('branch', 'branchname', 'character varying', 'NONE', None, None),
 ('branch', 'address', 'character varying', 'NONE', None, None),
 ('branch', 'ifsccode', 'character varying', 'NONE', None, None),
 ('crime_reports', 'dr_no', 'bigint', 'PK', None, None),
 ('crime_reports',
  'date_rptd',
  'timestamp without time zone',
  'NONE',
  None,
  None),
 ('crime_reports',
  'date_occ',
  'timestamp without time zone',
  'NONE',
  None,
  None),
 ('crime_reports', 'time_occ', 'character', 'NONE', None, None),
 ('crime_reports', 'a

In [16]:
batch_data

[{'table_name': 'crime_reports',
  'column_name': 'part_1_2',
  'data_type': 'smallint'},
 {'table_name': 'customer', 'column_name': 'dateofbirth', 'data_type': 'date'},
 {'table_name': 'customer',
  'column_name': 'incomelevel',
  'data_type': 'numeric'},
 {'table_name': 'customer', 'column_name': 'joindate', 'data_type': 'date'},
 {'table_name': 'accounts',
  'column_name': 'accountnumber',
  'data_type': 'integer'},
 {'table_name': 'accounts',
  'column_name': 'customerid',
  'data_type': 'integer'},
 {'table_name': 'accounts',
  'column_name': 'accountopendate',
  'data_type': 'date'},
 {'table_name': 'accounts', 'column_name': 'balance', 'data_type': 'numeric'},
 {'table_name': 'loans', 'column_name': 'loannumber', 'data_type': 'integer'},
 {'table_name': 'loans', 'column_name': 'customerid', 'data_type': 'integer'},
 {'table_name': 'loans',
  'column_name': 'sanctionedamount',
  'data_type': 'numeric'},
 {'table_name': 'loans',
  'column_name': 'interestrate',
  'data_type': 'num